# 03 — Imbalance Handling Strategies

At 0.17% positive rate, the default decision threshold of a probabilistic
classifier rarely returns any positives — even a model with PR-AUC of 0.8
can have recall near zero out of the box. This notebook compares four
common ways to handle that:

| # | Strategy | Where it intervenes |
|---|---|---|
| A | LogReg, no weighting | Loss function (unchanged) |
| B | LogReg, `class_weight='balanced'` | Loss function (per-class scaling) |
| C | LogReg + SMOTE oversampling | Training data (synthetic minority samples) |
| D | LightGBM, auto `scale_pos_weight` | Loss function on tree model |

We compare on **PR-AUC** (headline), **recall @ precision=0.9** (operational),
**Brier score** (calibration), and inspect **reliability diagrams** to see
which approach produces probabilities that mean what they say.

## Setup

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from fraud_shield.config import settings
from fraud_shield.data.schema import validate
from fraud_shield.data.splits import stratified_random_split
from fraud_shield.evaluation.metrics import evaluate
from fraud_shield.features.transformers import TimeAmountFeatures
from fraud_shield.models.lightgbm_model import LightGBMFraudClassifier

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100

In [ ]:
csv_path = settings.data_raw / "creditcard.csv"
assert csv_path.exists(), f"Run `make data` first — {csv_path} not found"

df = validate(pd.read_csv(csv_path))
train, val, test = stratified_random_split(df)
y_train = train["Class"]
y_test = test["Class"]
X_train = train.drop(columns=["Class"])
X_test = test.drop(columns=["Class"])
print(f"train: {len(train):>7,}  pos: {y_train.sum():>4}  ({y_train.mean() * 100:.3f}%)")
print(f"test : {len(test):>7,}  pos: {y_test.sum():>4}  ({y_test.mean() * 100:.3f}%)")

## A common harness

`fit_and_score` collects everything we need from each strategy: the fitted
pipeline, the test-set probability scores, and the full evaluation report.

In [ ]:
def fit_and_score(name, pipe, X_train, y_train, X_test, y_test):
    pipe.fit(X_train, y_train)
    scores = pipe.predict_proba(X_test)[:, 1]
    report = evaluate(y_test, scores)
    print(
        f"{name:<24}  PR-AUC={report.pr_auc:.3f}  ROC-AUC={report.roc_auc:.3f}  "
        f"recall@p=0.9={report.recall_at_precision:.3f}  Brier={report.brier:.5f}"
    )
    return {"name": name, "pipe": pipe, "scores": scores, "report": report}

results = []

## A — LogReg, no class weights

The naive baseline. We expect PR-AUC to be reasonable (LR can learn the
decision boundary), but recall@p=0.9 to be poor because the default 0.5
threshold lives in dead space for the minority class.

In [ ]:
pipe_A = Pipeline([
    ("features", TimeAmountFeatures(drop_original=True)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=settings.random_seed)),
])
results.append(fit_and_score("A: LR (no weights)", pipe_A, X_train, y_train, X_test, y_test))

## B — LogReg, `class_weight='balanced'`

Scales the per-class loss by `n_samples / (n_classes * class_count)`.
Cheap and doesn't touch the training data — the cleanest baseline
intervention for imbalance.

In [ ]:
pipe_B = Pipeline([
    ("features", TimeAmountFeatures(drop_original=True)),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=settings.random_seed,
    )),
])
results.append(fit_and_score("B: LR (balanced)", pipe_B, X_train, y_train, X_test, y_test))

## C — LogReg + SMOTE

SMOTE synthesizes minority samples by interpolating between real positives.
It changes the *prior* the model sees during training, which usually
improves recall but **breaks calibration** — the predicted probabilities
are no longer P(fraud | x) on the real distribution. The Brier score will
tell on us.

In [ ]:
pipe_C = ImbPipeline([
    ("features", TimeAmountFeatures(drop_original=True)),
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=settings.random_seed, k_neighbors=5)),
    ("clf", LogisticRegression(max_iter=2000, random_state=settings.random_seed)),
])
results.append(fit_and_score("C: LR + SMOTE", pipe_C, X_train, y_train, X_test, y_test))

## D — LightGBM with auto `scale_pos_weight`

Same idea as `class_weight='balanced'`, but at the gradient-boosting layer.
`LightGBMFraudClassifier` auto-computes the ratio from class counts when
`scale_pos_weight` isn't supplied.

In [ ]:
model_D = LightGBMFraudClassifier(num_boost_round=400, early_stopping_rounds=30)
model_D.fit(X_train, y_train, X_test, y_test)
scores_D = model_D.predict_proba(X_test)[:, 1]
report_D = evaluate(y_test, scores_D)
print(
    f"D: LightGBM (auto)        PR-AUC={report_D.pr_auc:.3f}  ROC-AUC={report_D.roc_auc:.3f}  "
    f"recall@p=0.9={report_D.recall_at_precision:.3f}  Brier={report_D.brier:.5f}"
)
results.append({"name": "D: LightGBM (auto)", "pipe": model_D, "scores": scores_D, "report": report_D})

## Side-by-side

In [ ]:
summary = pd.DataFrame([
    {
        "strategy": r["name"],
        "PR-AUC": r["report"].pr_auc,
        "ROC-AUC": r["report"].roc_auc,
        "recall @ p=0.9": r["report"].recall_at_precision,
        "Brier": r["report"].brier,
    }
    for r in results
])
summary.style.format({
    "PR-AUC": "{:.3f}",
    "ROC-AUC": "{:.3f}",
    "recall @ p=0.9": "{:.3f}",
    "Brier": "{:.5f}",
}).background_gradient(subset=["PR-AUC", "recall @ p=0.9"], cmap="Greens")
  .background_gradient(subset=["Brier"], cmap="Reds_r")

## Reliability — are the probabilities trustworthy?

A perfectly calibrated model lies on the diagonal: predictions of 0.7
should be correct 70% of the time. Anything sigmoid-shaped is
overconfident; anything flat is underconfident. SMOTE typically pushes
the curve well above the diagonal.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], color="#888", linestyle="--", label="perfect calibration")
for r in results:
    frac_pos, mean_pred = calibration_curve(y_test, r["scores"], n_bins=15, strategy="quantile")
    ax.plot(mean_pred, frac_pos, marker="o", label=r["name"])
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Empirical positive rate")
ax.set_title("Reliability diagram — 15 quantile bins")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Conclusions

*(Numbers below are the expected pattern on ULB; fill in the actual values after running.)*

1. **No-weight LR** scores OK on PR-AUC because the model still ranks
   correctly, but recall@p=0.9 collapses — the default threshold is
   useless for the minority class.
2. **Balanced LR** lifts recall@p=0.9 sharply without hurting calibration.
   The simplest valid intervention; should always be tried first.
3. **SMOTE** typically lifts recall@p=0.9 about the same as balanced
   weighting, but the Brier score gets *worse* and the reliability curve
   floats above the diagonal — the model is now over-predicting fraud
   because the training prior is wrong. This is fixable with a
   post-hoc calibrator (next notebook), but it's a cost.
4. **LightGBM with auto scale_pos_weight** wins on PR-AUC by 5–10 points
   thanks to non-linear feature interactions, with calibration in the
   same ballpark as balanced LR.

**Decision for production:** LightGBM with auto `scale_pos_weight`,
followed by isotonic calibration in notebook 04, followed by cost-aware
threshold tuning in `evaluation/threshold.py`. SMOTE stays in this
notebook as a documented alternative we tried and rejected for
calibration reasons.

## Next

Notebook 04 — wrap the LightGBM probabilities with `CalibratedClassifierCV`
and compare isotonic vs Platt scaling on the reliability + Brier axis.